## Load and Inspect the Data

In [ ]:
import pandas as pd
import sklearn

df = pd.read_csv('dataset.csv')

# df.head()
df.info()
# df.describe()

## Clean the dataset

### Handle timestamp

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")
df.head()

### Check for missing values

In [ ]:
df.isnull().sum()

### Flip target meaning, for 1 to mean SLA violation and 0 to mean SLA met

In [ ]:
df["target"].value_counts()


In [ ]:
df["target"] = 1 - df["target"]
df["target"].value_counts()
df.head()

## Feature Selection

In [41]:
features = [
 'task_size',
 'cpu_demand',
 'memory_demand',
 'network_latency',
 'io_operations',
 'disk_usage',
 'num_connections',
 'priority_level',
]

x = df[features]
y = df["target"]

## Data split into train. validation and split
In Chronological order

In [ ]:
train_size = int(len(df) * 0.7)
validation_size = int(len(df) * 0.15)
print(train_size, validation_size)
print(train_size+validation_size)

x_train = x[:train_size]
y_train = y[:train_size]

x_validation = x[train_size:train_size+validation_size]
y_validation = y[train_size:train_size+validation_size]

x_test = x[train_size+validation_size:]
y_test = y[train_size+validation_size:]

## Train with Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(x_train, y_train)

## Evaluate with validation set

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

validation_predictions = model.predict(x_validation)
validation_probabilities = model.predict_proba(x_validation)[:,1]

print(classification_report(y_validation, validation_predictions))
print("ROC-AUC: ", roc_auc_score(y_validation, validation_probabilities))

In [ ]:
threshold = 0.4
validation_custom = (validation_probabilities >= threshold).astype(int)

print(classification_report(y_validation, validation_custom))

In [ ]:
threshold = 0.35
validation_custom = (validation_probabilities >= threshold).astype(int)

print(classification_report(y_validation, validation_custom))

## Coefficients

In [16]:
import numpy as np

for feature, coef in zip(features, model.coef_[0]):
 print(feature, coef)

cpu_demand 1.144501654664211
memory_demand 0.04729080433716196
network_latency 0.009005554592450537
io_operations 0.3208009018712131
num_connections 0.009796333744292087


## Precision-Recall Curve

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

val_probs = model.predict_proba(x_validation)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_validation, val_probs)

plt.figure()
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

In [ ]:
import numpy as np

threshold = 0.2

idx = np.argmin(np.abs(thresholds - threshold))

plt.figure()
plt.plot(recall, precision)
plt.scatter(recall[idx], precision[idx])
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (with current threshold)")
plt.show()

In [ ]:
from sklearn.metrics import average_precision_score

ap = average_precision_score(y_validation, val_probs)
print("Average Precision:", ap)

In [ ]:
import numpy as np
target_recall = 0.98

idx = np.argmin(np.abs(recall - target_recall))

best_threshold = thresholds[idx]
best_precision = precision[idx]
best_recall = recall[idx]

print("Threshold:", best_threshold)
print("Precision:", best_precision)
print("Recall:", best_recall)